# 搜索与价值学习：已知规则的规划，未知价值的更新

先在非负权图上比较最少步数与最小成本，再在五格环境比较价值迭代与Q-learning。环境为人工构造，所有运行离线。

[经典AI](../../01-concepts/classical-ai/README.md) · [强化学习](../../01-concepts/reinforcement-learning/README.md) · [源码](../../05-code/foundations_core.py)。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '01-ai-foundations' / '05-code'))
from foundations_core import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. BFS最短步数可能不是最小成本

S-A-G是两步但成本9；S-B-C-G是三步但成本5。启发式是这个小图的真实剩余最短代价，专门用来说明A*怎样改变展开顺序，不是自动生成的启发式。

In [2]:
from collections import deque
graph={'S':[('A',1),('B',3)],'A':[('G',8)],'B':[('C',1)],'C':[('G',1)],'G':[]}
def bfs_path(graph,start,goal):
    queue=deque([(start,[start],0)])
    seen={start}
    while queue:
        state,path,cost=queue.popleft()
        if state==goal:return path,cost
        for nxt,edge in graph[state]:
            if nxt not in seen:
                seen.add(nxt);queue.append((nxt,path+[nxt],cost+edge))
    raise ValueError('unreachable')
h={'S':5,'A':8,'B':2,'C':1,'G':0}
bfs=bfs_path(graph,'S','G')
ucs=astar(graph,'S','G',lambda s:0)
a_star=astar(graph,'S','G',h.__getitem__)
print('BFS path,cost:',bfs)
print('UCS path,cost,expanded:',ucs)
print('A*  path,cost,expanded:',a_star)
assert bfs[1]==9 and ucs[1]==a_star[1]==5
assert a_star[2]<ucs[2]

BFS path,cost: (['S', 'A', 'G'], 9)
UCS path,cost,expanded: (['S', 'B', 'C', 'G'], 5.0, 5)
A*  path,cost,expanded: (['S', 'B', 'C', 'G'], 5.0, 4)


## 2. 验证“允许重新打开”的意义

此图的启发式可采纳但不一致：S-A 和 B-A 边违反一致性，而 S-B 边满足一致性。更便宜的S-B-A路径后到达，A应重新入队。配套代码没有简单地把第一次访问设成永久关闭。

一致性要求 $h(u)\le c(u,v)+h(v)$。代入本图：S-A 为 $4>3+0$，B-A 为 $3>1+0$；S-B 为 $4\le1+3$，不是违例边。重新打开后路径为 S-B-A-G，总成本 4。

In [3]:
reopen_graph={'S':[('A',3),('B',1)],'B':[('A',1)],'A':[('G',2)],'G':[]}
reopen_h={'S':4,'A':0,'B':3,'G':0}
path,cost,expanded=astar(reopen_graph,'S','G',reopen_h.__getitem__)
print('reopened result:',path,cost,'expanded:',expanded)
assert path==['S','B','A','G'] and cost==4
try:
    astar({'S':[]},'S','G',lambda s:0)
except ValueError as error: print('unreachable correctly rejected:',error)

reopened result: ['S', 'B', 'A', 'G'] 4.0 expanded: 5
unreachable correctly rejected: Goal unreachable


## 3. 五格环境的奖励只在进入终点时发放一次

位置0起步，左/右移动，进入4奖励1并结束，其余奖励0。理论最优值为[0.9³,0.9²,0.9,1,0]。价值迭代知道转移函数，Q-learning只用实际采样转移更新。

In [4]:
values=value_iteration()
q=q_learning()
expected=np.array([.9**3,.9**2,.9,1,0])
print('value iteration:',values)
print('Q table [left,right]:',q,sep='\n')
print('greedy policy (0 left,1 right):',np.argmax(q[:-1],axis=1))
print('max value error:',np.max(abs(q.max(axis=1)-expected)))
assert np.allclose(values,expected,atol=1e-10)
assert np.allclose(q.max(axis=1),expected,atol=1e-5)
assert np.all(np.argmax(q[:-1],axis=1)==1)
assert np.all(q[-1]==0)

value iteration: [0.729 0.81  0.9   1.    0.   ]
Q table [left,right]:
[[0.6561 0.729 ]
 [0.6561 0.81  ]
 [0.729  0.9   ]
 [0.81   1.    ]
 [0.     0.    ]]
greedy policy (0 left,1 right): [1 1 1 1]
max value error: 8.881784197001252e-16


## 4. 没有探索时可能永远看不到奖励

初始Q全0，固定argmax平局偏向左；epsilon=0时总从0向左，值无法改善。这个失败来自数据收集策略，并不需要换一个更大的模型。

In [5]:
no_explore=q_learning(episodes=100,epsilon=0)
print('Q without exploration:',no_explore,sep='\n')
assert np.all(no_explore==0)
state=0;trajectory=[]
for _ in range(10):
    action=int(np.argmax(q[state]))
    nxt,reward,done=line_transition(state,action)
    trajectory.append((state,action,reward,nxt,done))
    state=nxt
    if done:break
print('frozen greedy evaluation:',trajectory)
assert done and len(trajectory)==4

Q without exploration:
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]
frozen greedy evaluation: [(0, 1, 0.0, 1, False), (1, 1, 0.0, 2, False), (2, 1, 0.0, 3, False), (3, 1, 1.0, 4, True)]


## 观察与边界

UCS与A*找到同样最优成本，A*因好启发式少展开一个节点；BFS最少步但更贵。Q-learning在这个有限确定环境学到理论值，无探索版本失败。固定学习率、单个种子和简单环境不是一般收敛或复杂任务能力的证明。MCTS/CSP/POMDP仍是文章里的机制扩展，没有在本Notebook实现。

练习：让向右有20%概率原地不动；此时先改正确的转移/奖励模型，再分别修改价值迭代的期望和Q-learning的采样环境。